In [1]:
import pandas as pd
import numpy as np

In [2]:
df_cg = pd.read_csv('edges/chemgene.csv')
df_cd = pd.read_csv('edges/chemdis.csv')

df_c = pd.read_csv('vocab/CTD_chemicals.csv')

df_cg = df_cg[(df_cg['OrganismID'] == 9606)].drop_duplicates(subset=['ChemicalID', 'GeneSymbol'])
df_cd = df_cd.drop_duplicates(subset=['ChemicalID', 'DiseaseName'])  

In [3]:
chemicals = list(set(df_cg['ChemicalName'].unique()) | set(df_cd['ChemicalName'].unique()) | set(df_c['ChemicalName'].unique()))
len(chemicals)

179408

In [4]:
import requests
notfoundlist = []

def get_smiles(mesh_name):
    type = 'c'
    try:
        try:
            cid = requests.get(
                f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{mesh_name}/cids/JSON"
            ).json()["IdentifierList"]["CID"][0]
            # smiles = requests.get(
            #     f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/CanonicalSMILES/JSON"
            # ).json()['PropertyTable']['Properties'][0]['ConnectivitySMILES']
        except:
            type = 's'
            cid = requests.get(
                f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/substance/name/{mesh_name}/sids/JSON"
            ).json()["IdentifierList"]["SID"][0]
            # smiles = requests.get(
            #     f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/substance/sid/{cid}/property/CanonicalSMILES/JSON"
            # ).json()['PropertyTable']['Properties'][0]['ConnectivitySMILES']
    except:
        cid = None
        smiles = None
    # if not cid:

    # 
    # smiles = requests.get(
    #     f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/CanonicalSMILES/JSON"
    # ).json()
    if cid == None:
        notfoundlist.append(mesh_name)
    else:
        return cid,type


In [ ]:
import csv
from tqdm import tqdm
with open('chemid.csv','w',newline="") as chemfile:
    writer = csv.writer(chemfile)
    for chem in tqdm(chemicals):
        chemid = df_c.loc[df_c['ChemicalName'] == chem, 'ChemicalID'].iloc[0]
        smiles = get_smiles(chem)
        try:
            writer.writerow([chemid,smiles[0],smiles[1]])
            chemfile.flush()
        except:
            continue



  0%|          | 121/179408 [23:47<772:06:47, 15.50s/it]

In [6]:

with open('unfound.txt','w',newline="") as notfoundfile:
    notfoundfile.write(str(notfoundlist))


In [7]:
len(notfoundlist)

38

In [1]:

with open('unfound.txt','r',newline="") as notfoundfile:
    notfoundlist = notfoundfile.read()


In [7]:
print(notfoundlist)

['1-palmitoyl-2-(epoxycyclopentenone)-sn-glycero-3-phosphocholine', 'Fortasyn Connect', '2-(4-toluidino)-6-naphthalenesulfonic acid', 'N-(2-chloroethyl)-N-nitrosoacetamide', 'SR 142948A', 'antalarmin', 'lupalbigenin', 'zifrosilone', '16-(4-fluorophenoxy)lipoxin A4', 'Loteprednol Etabonate', 'cinnamic acid', 'ODN2006', 'leonurine', '3-ingenyl angelate', 'chromium trioxide', 'Technetium Tc 99m Sestamibi', 'Adenylyl Cyclase Inhibitors', 'Arsenites', 'Trenbolone Acetate', 'SSR180575', '3-(7-trifluoromethyl-5-(2-trifluoromethylphenyl)-1H-benzimidazol-2-yl)-1-oxa-2-azaspiro(4.5)dec-2-ene', "2,2',4,6'-tetramethoxystilbene", 'ReN 1869', 'terpendole E', 'alanylproline-4-nitroanilide', 'Furocoumarins', 'Tetraethyl Lead', '7,8-diacetoxy-4-methylthiocoumarin', 'carbonyl sulfide', 'Acyl Coenzyme A', '2-methoxyidazoxan', 'nitrosonium tetrafluoroborate', 'perfluorooctanesulfonamide', 'tepirindole', '(endo-N-8-methyl-8-azabicyclo-(3.2.1)oct-3-yl)-2,3-dihydro-3-isopropyl-2-oxo-1H-benzimidazol-1-carboxa

In [1]:
import pandas as pd

cids = pd.read_csv('chemids.csv')
print(cids)
chemIDs = list(cids['chemID'])
print(chemIDs)

                chemID       cids type
0         MESH:D012378  162064824    s
1         MESH:C070055      35823    c
2         MESH:C024713      12389    c
3      MESH:D000077612     166548    c
4         MESH:C538809  131634859    c
...                ...        ...  ...
17752     MESH:C475761    1816834    c
17753     MESH:C030233      18300    c
17754  MESH:C000607970      42640    c
17755     MESH:D003983       5820    c
17756     MESH:C485003   10212249    c

[17757 rows x 3 columns]
['MESH:D012378', 'MESH:C070055', 'MESH:C024713', 'MESH:D000077612', 'MESH:C538809', 'MESH:C000626479', 'MESH:C104428', 'MESH:C066686', 'MESH:C570403', 'MESH:C071363', 'MESH:C005277', 'MESH:C024221', 'MESH:C553923', 'MESH:C096294', 'MESH:C553921', 'MESH:C018209', 'MESH:C077369', 'MESH:C100095', 'MESH:C060297', 'MESH:D020355', 'MESH:D000077464', 'MESH:C086427', 'MESH:C477983', 'MESH:C006821', 'MESH:C412945', 'MESH:C523198', 'MESH:C054931', 'MESH:C402769', 'MESH:C013997', 'MESH:C113498', 'MESH:C524315', 

In [2]:
import requests
notfoundlist = []

def get_smiles(cid,typeof):
    typeof = 'c'
    try:
        if typeof == 'c':            
            smiles = requests.get(
                f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/CanonicalSMILES/JSON"
            ).json()['PropertyTable']['Properties'][0]['ConnectivitySMILES']
        else:
            smiles = requests.get(
                f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/substance/sid/{cid}/property/CanonicalSMILES/JSON"
            ).json()['PropertyTable']['Properties'][0]['ConnectivitySMILES']
    except:
        smiles = None
    if smiles == None:
        notfoundlist.append(cid)
    else:
        return smiles


In [ ]:
import csv
from tqdm import tqdm
import pandas as pd

cids = pd.read_csv('chemids.csv')
chemIDs = list(cids['chemID'])
types = list(cids['type'])
cidlist = list(cids['cids'])
with open('mesh_to_smiles.csv','w',newline="") as chemfile:
    writer = csv.writer(chemfile)
    chemIDS = cids['chemID']
    print(chemIDS)
    for i in tqdm(range(len(types))):
        chemid = chemIDS[i]
        typeof = types[i]
        cid = cidlist[i]
        smiles = get_smiles(cid,typeof)
        try:
            writer.writerow([chemid,smiles])
            chemfile.flush()
        except:
            continue

0           MESH:D012378
1           MESH:C070055
2           MESH:C024713
3        MESH:D000077612
4           MESH:C538809
              ...       
17752       MESH:C475761
17753       MESH:C030233
17754    MESH:C000607970
17755       MESH:D003983
17756       MESH:C485003
Name: chemID, Length: 17757, dtype: object


 13%|█▎        | 2267/17757 [55:53<6:39:06,  1.55s/it] 

In [ ]:
with open('unfoundsmiles.txt','w',newline="") as notfoundfile:
    notfoundfile.write(str(notfoundlist))